# Session 09: Privacy and East African law

Find and mask personal data in Kitara's claims letters, delete one customer's letter from the index, and look at the customer table. Then draw your own data protection map in Data Inventory.xlsx and the page 'Data protection map' of the Participant Workbook.

**Student version.** This is the team notebook. Leaders can follow it on the shared screen. Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("customers.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith(".csv")), "and the docs folder")

## Setup: load the documents
This cell reads the Kitara documents, keeps the 32 flagged as holding personal data and picks the 30 text letters. It takes about twenty seconds because it reads the scans with OCR. Run it once.

In [ ]:
import re
import pandas as pd
import kitara_search as ks

corpus = ks.load_corpus('.')
flag = corpus[corpus.contains_personal_data == 'Y']
letters = flag[flag.format == 'text_pdf']
print(len(corpus), 'documents,', len(flag), 'flagged,', len(letters), 'text letters')

## Task 1. Count what the patterns find
The dictionary `PATTERNS` holds five regular expressions. Fill in the national id, customer id and email patterns from the slides. Write `count_matches(texts, rx)` that returns two numbers: the matches, and the number of texts with at least one. Build `counts` for the five patterns on `letters.text`.

In [ ]:
PATTERNS = {
    'PHONE': r'\+25[46] ?700 ?0\d\d ?\d{3}',
    # Your turn: NATIONAL_ID is the letters KX and 12 digits
    # Your turn: CUSTOMER_ID is KT, a hyphen and 7 digits
    # Your turn: EMAIL is word characters or dots, an at sign, then a domain
    'NAME': r'\b(?:Mr|Mrs|Ms|Dr) [A-Z]\w+(?: [A-Z]\w+)*',
}

def count_matches(texts, rx):
    # Your turn: use texts.str.count(rx), then return the sum and the number of texts above 0
    pass

# Your turn: counts = a dictionary with one (matches, letters) pair for each pattern


In [ ]:
_missing = [v for v in ('counts',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert counts['PHONE'] == (49, 26), 'Phone numbers: 49 matches in 26 letters'
assert counts['NATIONAL_ID'] == (20, 20), 'National ids: 20 matches in 20 letters'
assert counts['CUSTOMER_ID'] == (30, 30), 'Customer ids: 30 matches in 30 letters'
assert counts['EMAIL'] == (0, 0), 'The letters hold no emails'
assert counts['NAME'] == (37, 30), 'Titled names: 37 matches in 30 letters'
print('Task 1 correct')

## Task 2. Mask the letters
Write `mask(text, patterns)`. It first finds each titled name and replaces it, and the same name without its title, with `[NAME]`. Then it replaces every pattern with a token carrying its label, such as `[PHONE]`. Build `masked` as the masked text of every letter and count the tokens in `token_counts`.

In [ ]:
def mask(text, patterns):
    # Your turn: for each titled name from re.findall(patterns['NAME'], text), replace the full
    #            name, and the name without its title, with '[NAME]'
    # Your turn: for every pattern use re.sub to replace each match with '[' + the pattern's key + ']'
    return text

masked = letters.text.map(lambda t: mask(t, PATTERNS))
# Your turn: token_counts = how often each token appears in masked
#            (str.extractall with r'\[([A-Z_]+)\]', then value_counts().to_dict())


In [ ]:
_missing = [v for v in ('masked', 'token_counts',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert token_counts['PHONE'] == 49 and token_counts['NATIONAL_ID'] == 20 and token_counts['CUSTOMER_ID'] == 30, 'Expected 49 PHONE, 20 NATIONAL_ID and 30 CUSTOMER_ID tokens'
assert not masked.str.contains(r'KX\d{12}|KT-\d{7}|\+25[46] ?700').any(), 'A pattern is still in the masked text'
print('Task 2 correct')

## Task 3. A check that finds what the masker missed
The two scans in `flag` were read with OCR and carry spelling slips. Mask all 32 flagged documents with `PATTERNS`, cut them into chunks with `ks.build_chunks`, and write `leaks(table)` that returns the sorted chunk ids still matching any pattern in `LOOSE`. Store the result in `leaked`. Then mask again with `MASKER = dict(PATTERNS, **LOOSE)` and store the new result in `leaked_after`.

In [ ]:
LOOSE = {
    'PHONE': r'\+25[46] ?(?:[\dO@]{3} ?){2}[\dO@]{3}',
    'NATIONAL_ID': r'(?i)kx ?\d{12}',
    'CUSTOMER_ID': r'(?i)kt-\d{7}',
    'EMAIL': PATTERNS['EMAIL'],
}

def leaks(table):
    # Your turn: collect the chunk_id of every row whose text matches any pattern in LOOSE
    #            (table.text.str.contains(rx)), and return the ids as a sorted list
    pass

# Your turn: safe = flag with its text masked using PATTERNS; leaked = leaks(ks.build_chunks(safe))
# Your turn: MASKER = dict(PATTERNS, **LOOSE); mask again; leaked_after = leaks(...) on the new chunks


In [ ]:
_missing = [v for v in ('leaked', 'leaked_after',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert leaked == ['D084-01', 'D087-01'], 'The two scanned documents should leak: D084-01 and D087-01'
assert leaked_after == [], 'With the looser patterns in the masker nothing should leak'
print('Task 3 correct')

## Task 4. Erase one customer from the index
The customer with number `KT-8782200` asks for deletion. Build the chunks of the whole `corpus`, find the chunks that hold that number, remove every chunk of those documents from the table, and rebuild a `ks.Searcher`. Search for `dialysis twice a week kidney failure` (hybrid, top 3, `user_groups='claims'`) before and after. Store the two lists of chunk ids in `before` and `after`.

In [ ]:
chunks = ks.build_chunks(corpus)
search = ks.Searcher(chunks)
q = 'dialysis twice a week kidney failure'
# Your turn: before = the chunk ids of the top 3 hybrid results for q, with user_groups='claims'
# Your turn: hit = the chunks whose text contains 'KT-8782200'
# Your turn: remove every chunk whose doc_id is in hit.doc_id, reset the index, and rebuild search
# Your turn: after = the top 3 chunk ids again


In [ ]:
_missing = [v for v in ('after', 'before', 'chunks',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert before[0] == 'D102-01', 'Before the deletion Letter 11 (D102-01) should come first'
assert 'D102-01' not in after, 'After the deletion no chunk of D102 may be returned. Rebuild the Searcher from the smaller table'
assert len(chunks) == 138, 'One chunk should have been removed, leaving 138'
print('Task 4 correct')

## Task 5. The customer table and data minimisation
Read `customers.csv` as text. Build `filled`, the number of non-blank values in each column of `PERSONAL`. Build `keep`, the four columns `product`, `segment`, `district` and `status`. Count the national ids that match `KX` and 12 digits in `kx_ok`.

In [ ]:
cust = pd.read_csv('customers.csv', dtype=str)
PERSONAL = ['full_name', 'date_of_birth', 'phone', 'email', 'national_id']
# Your turn: filled = the count of non-blank values in each PERSONAL column, as a dictionary
# Your turn: keep = cust with only product, segment, district and status
# Your turn: kx_ok = how many non-blank national_id values match KX followed by 12 digits


In [ ]:
_missing = [v for v in ('filled', 'keep', 'kx_ok',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert filled == {'full_name': 5000, 'date_of_birth': 4700, 'phone': 5000, 'email': 3377, 'national_id': 4450}, 'Check the counts of non-blank values'
assert list(keep.columns) == ['product', 'segment', 'district', 'status'], 'Keep four columns'
assert kx_ok == 4450, 'Every filled national id follows the KX pattern: 4450'
print('Task 5 correct')

## Optional: Presidio
Presidio is not installed in the course notebook. On your own machine, install `presidio-analyzer` and `presidio-anonymizer` and download the language model named in its documentation first. The cell below shows the simplest form. Detection is model based and never complete, so keep the check from Task 3.

In [ ]:
# pip install presidio-analyzer presidio-anonymizer
# Your turn: import AnalyzerEngine and AnonymizerEngine, analyze the first letter in English,
#            and print the entity types and scores


## For the leader
Look at your results. The masked letters pass the check, yet they still carry a name without a title, a policy number and a health detail. Who in your organisation would sign off that masked letters may enter a search index, and who signs off that any personal data may be stored in a cloud region in another country? Write the two names, or write the gap, below.